# ONNX Exercise: ONNX meets FastAPI

Now that you've learned about ONNX and FastAPI, let's try to use them.

Our goal: We need a REST API that predicts, whether an applicant should be granted a loan (or not). We have already trained a model for this (please run `onnx_introduction.ipynb` in before), which is available at `model/loan_model.onnx`. You are welcome to tweak the model quality, if you're keen to, but this is not subject of this exercise.

We have already started this task in the following cell, that contains part of the required code. You will find spots marked with `...`. In these spots you need to add your code.

In [4]:
%%writefile onnx_exercise.py
import numpy as np
import onnxruntime as rt
from fastapi import FastAPI
from pydantic import BaseModel
import onnx

# create the fastapi application
api = FastAPI()

# load the onnx model. for our demo purposes, the model may live in the global context.
# for more advanced apis, it may be reasonable to move the model into a dependency
sess = rt.InferenceSession("./loan_model.onnx")


# define the features that our request expects
class LoanFeatures(BaseModel):
    applicantincome: float
    coapplicantincome: float
    loanamount: float
    loan_amount_term: float
    credit_history: float
    married: str
    dependents: str
    education: str
    self_employed: str
    property_area: str
    

    @classmethod
    def get_type_fields(cls, _type) -> set:
        """Utility function to get all features of this BaseModel of a certain type.

        Example usage:
        `all_int_fields = LoanFeatures.get_type_fields(int)`  # result: {}
        """
        return {
            field for field, fieldinfo in cls.model_fields.items()
            if fieldinfo.annotation == _type
        }


# define the features that our response should send
class Prediction(BaseModel):
    label: str
    probability: float


# create a POST endpoint that retrieves prediction requests, performs the prediction and
# returns the prediction result as its response
@api.post("/predict")
def post_predict(loan_features: LoanFeatures) -> list[Prediction]:
    # convert the request object into a dict of numpy arrays
    # HINT: you can retrieve a dict with all features via loan_features.model_dump()
    loan_feats = {k: np.array(v) for k, v in loan_features.model_dump().items()}

    # convert the feature dtypes into the same dtypes that the model expects
    # NOTE that the input_value is expected to be two-dimensional, so we need to reshape it properly
    for c in LoanFeatures.get_type_fields(float):
        ins[c] = ins[c].astype(np.float32).reshape((1, 1))
    for k in LoanFeatures.get_type_fields(str):
        ins[k] = ins[k].reshape((1, 1))

    # run the prediction
    labels, probabilities = sess.run(None, ins)

    # create the response object
    predicted = []
    for k, v in probabilities[0].items():
        predicted.append(Prediction(label=k, probability=v))

    # return the response
    return predicted


Overwriting onnx_exercise.py


When you execute that cell, a file called `onnx_exercise.py` will be written. You can then serve the API with the following command:

In [5]:
!fastapi dev onnx_exercise.py --port 7777 --host 0.0.0.0


   FastAPI   Starting development server 🚀
 
             Searching for package file structure from directories with 
             __init__.py files
             Importing from /workshop/notebooks/onnx
 
    module   🐍 onnx_exercise.py
 
      code   Importing the FastAPI app object from the module with the following
             code:
 
             from onnx_exercise import api
 
       app   Using import string: onnx_exercise:api
 
    server   Server started at ]8;id=293112;http://0.0.0.0:7777\http://0.0.0.0:7777]8;;\
    server   Documentation at ]8;id=724556;http://0.0.0.0:7777/docs\http://0.0.0.0:7777/docs]8;;\
 
       tip   Running in development mode, for production use: fastapi run
 
             Logs:
 
      INFO   Will watch for changes in these directories: 
             ['/workshop/notebooks/onnx']
      INFO   Uvicorn running on http://0.0.0.0:7777 (Press CTRL+C to quit)
      INFO   Started reloader process [130] using WatchFiles
      INFO   Started server p